# 🕵️‍♂️ Student Notebook: Sherlock Holmes AI (Seq2Seq)

# 📘 Assignment: Text Continuation with Encoder-Decoder LSTM

**Objective:** Build a Sequence-to-Sequence (Seq2Seq) model that learns the style of Arthur Conan Doyle.
**Task:** Instead of simple next-word prediction, you will implement a **Hierarchical Encoder-Decoder** architecture to read a context and generate a continuation.

**Core Architecture:**
1.  **Encoder:** Reads the first half of a sequence (Context) and compresses it into a hidden state.
2.  **Decoder:** Takes the Encoder's hidden state and generates the second half of the sequence (Continuation).

In [ ]:
# @title 1. Setup & Clean Environment
import shutil
import os

if os.path.exists('/content/sherlock_project/processed_data'):
    shutil.rmtree('/content/sherlock_project/processed_data')
    print("🧹 Old processed data cleaned.")

if os.path.exists('/content/sherlock_project/checkpoints'):
    shutil.rmtree('/content/sherlock_project/checkpoints')
    print("🧹 Old checkpoints cleaned.")

### 1. Setup & Data Extraction

Mount your Google Drive and extract the dataset. Ensure the `sherlock_project.zip` file is present in your Drive root.

In [2]:
# @title Mount & Extract
from google.colab import drive
import os
import zipfile
import shutil
import sys

drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/sherlock_project.zip'
WORK_DIR = '/content/sherlock_project'

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

if not os.path.exists(ZIP_PATH):
    print(f"Error: {ZIP_PATH} not found.")
else:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(WORK_DIR)
    print("Project extracted.")

os.chdir(WORK_DIR)
sys.path.append(WORK_DIR)

Mounted at /content/drive
Project extracted.


###  Install Requirements

Now, let's install the necessary Python libraries. We need `torch` for the AI model, `nltk` for text processing, and `tqdm` for the progress bars.

In [3]:
# @title Install Requirements
!pip install -r requirements.txt -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.4/755.4 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.

### 2. Task 1: Preprocessing Pipeline

**Context:** `src/preprocessing.py`

Before training, raw text must be cleaned and tokenized.
**Your Objectives:**
1.  **`clean_text`**: Normalize case, handle punctuation, and remove noise.
2.  **`build_vocab`**: Create `Token -> ID` mappings.
    * *Critical:* You must include special tokens: `<PAD>=0`, `<SOS>=1`, `<EOS>=2`, `<UNK>=3`.
3.  **Pipeline**: Connect loading, cleaning, and encoding steps.

> **Note:** The quality of your generation depends heavily on the cleanliness of your vocabulary.

In [4]:
# @title Task 1: src/preprocessing.py
%%writefile src/preprocessing.py
import os
import re
import json
import torch
import collections
from typing import Tuple, Dict, List
from src import utils
from src.config import ProjectConfig


def combine_and_load_raw_data(raw_dir: str, output_path: str) -> str:
    """
    Provided helper: combines all .txt files into a single corpus.
    """
    if os.path.exists(output_path):
        with open(output_path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()

    combined_text = []
    for root, _, files in os.walk(raw_dir):
        for file in files:
            if file.endswith(".txt") and file != "sherlock.txt":
                try:
                    with open(os.path.join(root, file), 'r', encoding='utf-8', errors='ignore') as infile:
                        combined_text.append(infile.read())
                except:
                    pass

    full_text = "\n".join(combined_text)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(full_text)

    return full_text


def clean_text(text: str) -> str:
    """
    Normalize text:
    - lowercase
    - keep basic punctuation
    - remove non-standard characters
    - normalize whitespace
    """
    text = text.lower()

    # Keep letters, numbers, and common punctuation
    text = re.sub(r"[^a-z0-9\s\.\,\!\?\;\:\'\-]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def build_vocab(tokens: List[str], max_size: int, min_freq: int) -> Tuple[Dict[str, int], Dict[int, str]]:
    """
    Build vocabulary mappings.
    Must include:
    <PAD>=0, <SOS>=1, <EOS>=2, <UNK>=3
    """
    counter = collections.Counter(tokens)

    # Special tokens (fixed IDs)
    token_to_id = {
        "<PAD>": 0,
        "<SOS>": 1,
        "<EOS>": 2,
        "<UNK>": 3
    }

    # Reserve space for special tokens
    max_size = max_size - len(token_to_id)

    # Sort by frequency then alphabetically (stable, deterministic)
    sorted_tokens = sorted(
        [(tok, freq) for tok, freq in counter.items() if freq >= min_freq],
        key=lambda x: (-x[1], x[0])
    )

    for token, _ in sorted_tokens[:max_size]:
        if token not in token_to_id:
            token_to_id[token] = len(token_to_id)

    id_to_token = {idx: tok for tok, idx in token_to_id.items()}

    return token_to_id, id_to_token


def run_preprocessing_pipeline(config: ProjectConfig):
    logger = utils.get_logger("preprocessing")
    logger.info("Starting Preprocessing...")

    raw_dir = utils.RAW_DATA_DIR
    combined_path = os.path.join(raw_dir, "sherlock.txt")

    # 1. Load & clean text
    raw_text = combine_and_load_raw_data(raw_dir, combined_path)
    clean_corpus = clean_text(raw_text)

    # 2. Tokenize
    tokens = clean_corpus.split()

    # 3. Build vocabulary
    token_to_id, id_to_token = build_vocab(
        tokens,
        max_size=config.max_vocab_size,
        min_freq=2
    )

    # 4. Encode tokens
    encoded = [
        token_to_id.get(tok, token_to_id["<UNK>"])
        for tok in tokens
    ]

    # 5. Convert to tensor
    encoded_tensor = torch.tensor(encoded, dtype=torch.long)

    # 6. Save artifacts
    os.makedirs(os.path.dirname(utils.SEQUENCES_PATH), exist_ok=True)
    os.makedirs(os.path.dirname(utils.VOCAB_PATH), exist_ok=True)

    torch.save(encoded_tensor, utils.SEQUENCES_PATH)

    with open(utils.VOCAB_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {
                "token_to_id": token_to_id,
                "id_to_token": id_to_token
            },
            f,
            indent=2
        )

    logger.info(f"Vocabulary size: {len(token_to_id)}")
    logger.info(f"total tokens: {len(tokens)}")
    logger.info("Preprocessing pipeline finished.")


Overwriting src/preprocessing.py


### 3. Task 2: Dataset & Split Logic

**Context:** `src/dataset.py`

**Crucial Change:** Unlike standard Language Models that use a sliding window of 1 token, an Encoder-Decoder requires distinct **Source** and **Target** sequences.

**Your Logic:**
For a given data chunk of length `N` (e.g., 20 words):
* **Source (`src`):** The first `N/2` words. (Input to Encoder)
* **Target (`trg`):** The remaining `N/2` words. (Input/Target for Decoder)

**Implementation:**
* Implement `__getitem__` to return this `(src, trg)` pair.
* Ensure your sequences do not overlap incorrectly.

In [6]:
# @title Task 2: src/dataset.py
%%writefile src/dataset.py
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import os
from src import utils
from src.config import TrainConfig, DataConfig


class SherlockDataset(Dataset):
    """
    Each sample is a contiguous chunk of length seq_length.
    The chunk is split into:
      - src: first half  (encoder input)
      - trg: second half (decoder input / target)
    """

    def __init__(self, data_tensor: torch.Tensor, sequence_length: int):
        if sequence_length % 2 != 0:
            raise ValueError("Sequence length must be even for encoder-decoder split.")

        self.data = data_tensor
        self.sequence_length = sequence_length
        self.half = sequence_length // 2

        # Number of non-overlapping chunks
        self.n_samples = len(self.data) // self.sequence_length

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(self, idx: int):
        start = idx * self.sequence_length
        end = start + self.sequence_length

        chunk = self.data[start:end]

        # Split into source and target
        src = chunk[:self.half]
        trg = chunk[self.half:]

        return src, trg


def create_dataloaders(train_config: TrainConfig, data_config: DataConfig):
    """
    Creates train and validation dataloaders from the processed tensor.
    """

    if not os.path.exists(utils.SEQUENCES_PATH):
        raise FileNotFoundError("Processed data missing. Run preprocessing first.")

    data_tensor = torch.load(utils.SEQUENCES_PATH)

    dataset = SherlockDataset(data_tensor, data_config.sequence_length)

    # Train / validation split
    val_size = int(len(dataset) * 0.2) # val_split = 0.2
    train_size = len(dataset) - val_size

    train_dataset, val_dataset = random_split(
        dataset, [train_size, val_size]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=train_config.batch_size,
        shuffle=True,
        drop_last=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=train_config.batch_size,
        shuffle=False,
        drop_last=True
    )

    return train_loader, val_loader


Overwriting src/dataset.py


src (Source): The first half. This is fed into the Encoder to understand the context

trg (Target): The second half. This is what the Decoder is trained to predict as the continuation.

Train/Validation Split: It reserves 20% of the data for validation. The model never "sees" this data during training; it is only used to test if the model is actually learning or just memorizing (overfitting).

what happens :

Raw Tensor: [1, 54, 23, 10, 8, 99, 4, 12, ...]

SherlockDataset: Groups them into chunks of 100 tokens.

Split: Returns src (tokens 0-49) and trg (tokens 50-99).

DataLoader: Grabs 32 of these pairs, stacks them into a matrix, and sends them to the model.

### 4. Task 3: The Encoder-Decoder Model

**Context:** `src/model.py`

**Requirement:** Do **not** use a single monolithic LSTM. You must implement a modular Seq2Seq architecture.

**Your Architecture:**
1.  **`self.encoder_lstm`**: Processes the `src` sequence. Returns the final `hidden_state`.
2.  **`self.decoder_lstm`**: Initializes with the Encoder's `hidden_state`. Processes the `trg` sequence.
3.  **`forward(src, trg)`**: Manually bridge the two components.
    * Pass `src` through Encoder.
    * Transfer `(hidden, cell)` from Encoder to Decoder.
    * Pass `trg` through Decoder.

In [7]:
# @title Task 3: src/model.py
%%writefile src/model.py
import torch
import torch.nn as nn


class EncoderDecoder(nn.Module):
    """
    LSTM-based Encoder–Decoder (Seq2Seq) model.
    Encoder processes src sequence and produces context (hidden, cell).
    Decoder generates outputs conditioned on encoder context.
    """

    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        num_layers: int = 2,
        dropout_prob: float = 0.5,
    ):
        super().__init__()
        self.vocab_size = vocab_size

        # 1. Shared embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 2. Encoder LSTM
        self.encoder_lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_prob if num_layers > 1 else 0.0,
        )

        # 3. Decoder LSTM
        self.decoder_lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_prob if num_layers > 1 else 0.0,
        )

        # 4. Output projection
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(self, src: torch.Tensor, trg: torch.Tensor):
        """
        src: (B, src_len)
        trg: (B, trg_len)

        Returns:
            logits: (B, trg_len, vocab_size)
            (hidden, cell): final decoder states
        """

        # ---- Encoder ----
        # Embed source
        src_embedded = self.embedding(src)  # (B, src_len, emb_dim)

        # Encode
        _, (hidden, cell) = self.encoder_lstm(src_embedded)
        # hidden, cell: (num_layers, B, hidden_size)

        # ---- Decoder ----
        # Embed target
        trg_embedded = self.embedding(trg)  # (B, trg_len, emb_dim)

        # Decode using encoder context as initial state
        decoder_outputs, (hidden, cell) = self.decoder_lstm(
            trg_embedded, (hidden, cell)
        )
        # decoder_outputs: (B, trg_len, hidden_size)

        # ---- Projection ----
        logits = self.fc_out(decoder_outputs)
        # logits: (B, trg_len, vocab_size)

        return logits, (hidden, cell)


Overwriting src/model.py


self.embedding: Converts word IDs (integers) into dense vectors (floats).

self.encoder_lstm : It processes the prompt word-by-word and compresses the entire meaning into a final hidden state

self.decoder_lstm: It starts with the information compressed by the encoder and tries to unfold it into a new sequence of words.

### 5. Task 4: Training Loop

**Context:** `src/train.py`

Implement the standard PyTorch training loop with one key difference:
* Your model's `forward` method now accepts **two** arguments: `model(src, trg)`.
* Ensure you handle the Loss calculation by flattening the outputs to `(Batch * Seq_Len, Vocab_Size)`.

In [8]:
# @title Task 4: src/train.py
%%writefile src/train.py
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import os
from src.config import ProjectConfig
from src import utils
from src.dataset import create_dataloaders
from src.model import EncoderDecoder

class ModelTrainer:
    def __init__(self, config: ProjectConfig):
        self.config = config
        self.device = utils.get_device()
        self.logger = utils.get_logger("trainer")

        # Load data
        self.train_loader, self.val_loader = create_dataloaders(config.train, config.data)

        # Initialize Model
        self.model = EncoderDecoder(
            vocab_size=config.data.max_vocab_size,
            embedding_dim=config.model.embedding_dim,
            hidden_size=config.model.lstm_units,
            num_layers=config.model.num_layers,
            dropout_prob=config.model.dropout_rate
        ).to(self.device)

        self.optimizer = optim.Adam(self.model.parameters(), lr=config.train.learning_rate)
        # ignore_index=0 ensures we don't calculate loss on <PAD> tokens
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)

    def train_one_epoch(self, epoch_index: int):
        self.model.train()
        total_loss = 0
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch_index} [Train]", leave=False)

        for src, trg in pbar:
            src, trg = src.to(self.device), trg.to(self.device)

            # 1. Zero Grad
            self.optimizer.zero_grad()

            # 2. Forward pass
            # Note: Decoder uses trg as input for teacher forcing
            outputs, _ = self.model(src, trg)

            # 3. Calculate Loss
            # outputs: (Batch, Seq_Len, Vocab_Size) -> (Batch * Seq_Len, Vocab_Size)
            # trg: (Batch, Seq_Len) -> (Batch * Seq_Len)
            V = outputs.size(-1)
            loss = self.criterion(outputs.view(-1, V), trg.view(-1))

            # 4. Backward & Step
            loss.backward()

            # Gradient Clipping (prevents exploding gradients in LSTMs)
            if self.config.train.grad_clip_norm:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.train.grad_clip_norm)

            self.optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item())

        return total_loss / len(self.train_loader)

    def validate(self, epoch_index: int):
        self.model.eval()
        total_loss = 0
        pbar = tqdm(self.val_loader, desc=f"Epoch {epoch_index} [Val]", leave=False)

        with torch.no_grad():
            for src, trg in pbar:
                src, trg = src.to(self.device), trg.to(self.device)

                outputs, _ = self.model(src, trg)
                V = outputs.size(-1)
                loss = self.criterion(outputs.view(-1, V), trg.view(-1))

                total_loss += loss.item()
                pbar.set_postfix(loss=loss.item())

        return total_loss / len(self.val_loader) if len(self.val_loader) > 0 else 0

    def train(self):
        os.makedirs(utils.CHECKPOINT_DIR, exist_ok=True)
        self.config.save(os.path.join(utils.CHECKPOINT_DIR, 'config.json'))

        best_val_loss = float('inf')
        patience_counter = 0

        for epoch in range(1, self.config.train.epochs + 1):
            train_loss = self.train_one_epoch(epoch)
            val_loss = self.validate(epoch)

            self.logger.info(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

            state = {
                'epoch': epoch,
                'model_state_dict': self.model.state_dict(),
                'optimizer_state_dict': self.optimizer.state_dict(),
                'val_loss': val_loss
            }

            # Regular checkpoint
            if epoch % self.config.train.ckpt_every_n_epochs == 0:
                torch.save(state, os.path.join(utils.CHECKPOINT_DIR, f'checkpoint_{epoch}.pth'))

            # Save best model and handle Early Stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(state, os.path.join(utils.CHECKPOINT_DIR, 'best_model.pth'))
                self.logger.info(f"⭐ New best model saved at epoch {epoch}")
            else:
                patience_counter += 1

            if self.config.train.early_stop_patience and patience_counter >= self.config.train.early_stop_patience:
                self.logger.info(f"🛑 Early stopping triggered at epoch {epoch}")
                break


Overwriting src/train.py


ignore_index=0 : empty padding <PAD> 0 reward.

what happens:

Load 32 sentences.

Predict the second half of those sentences.

Calculate how far off those predictions were (Loss).

Update the LSTM weights to be slightly more accurate.

Check the "Unseen" validation set to see if the model is actually getting smarter.

Repeat until the model stops improving.

### 6: Hyperparameter Tuning & Configuration

Now that your pipeline is built, you have full control over the AI's "Brain".
Below are the default settings. You can—and should—modify them to improve performance or solve overfitting.

### ⚠️ CRITICAL NOTE: Data Synchronization

These settings control how data is processed and how the model is built.
**If you change `max_vocab_size` or `seq_length`:**

1.  You **MUST** re-run the **Preprocessing Stage** (`!python main.py --stage preprocess`).
2.  If you skip preprocessing, the Training script will detect the mismatch and fallback to the old data's vocabulary size (as seen in your logs: `⚠️ Config says X, but data has Y`).

**Tuning Guide:**

  * **Overfitting?** (Repeating words like "night night") $\rightarrow$ Reduce `lstm_units` (e.g., 512 $\to$ 128), Reduce `num_layers`, or Increase `dropout_rate`.
  * **Underfitting?** (Gibberish text) $\rightarrow$ Train for more `epochs` or increase `embedding_dim`.

In [9]:
# @title ⚙️ Student Configuration Panel
from src.config import ProjectConfig
import os


if os.path.exists('checkpoints/config.json'):
    try: os.remove('checkpoints/config.json')
    except: pass

config = ProjectConfig()

# ============================================================
#  DATA SETTINGS
#  IF YOU CHANGE THESE, YOU MUST RE-RUN 'PREPROCESS' STAGE!
# ============================================================
config.data.seq_length = 100        # Total: 10 (Encoder) + 10 (Decoder). Must be EVEN.
config.data.max_vocab_size = 10000    # Shrink vocab to make learning easier for small data.

# ============================================================
#  MODEL ARCHITECTURE
# Adjust these to solve Overfitting (Repetition)
# ============================================================
config.model.embedding_dim = 128
config.model.lstm_units = 256
config.model.num_layers = 2
config.model.dropout_rate = 0.5

# ============================================================
# TRAINING LOOP
# ============================================================
config.train.epochs = 3
config.train.early_stop_patience = 3
config.train.batch_size = 32
config.train.learning_rate = 0.001
print (config.data.max_vocab_size)

# --- Auto-Correction & Save ---
if config.data.seq_length % 2 != 0:
    config.data.seq_length += 1
    print(f"🔧 Auto-corrected seq_length to even number: {config.data.seq_length}")

os.makedirs('checkpoints', exist_ok=True)
config.save('checkpoints/config.json')

print("✅ Configuration Saved!")
print(f"   🔹 Architecture: {config.model.num_layers} Layer(s), {config.model.lstm_units} Units")
print(f"   ⚠️ REMINDER: If you changed 'vocab_size' or 'seq_length', run PREPROCESS again!")
print(config.train.epochs)



10000
[ProjectConfig] Saved configuration to checkpoints/config.json
✅ Configuration Saved!
   🔹 Architecture: 2 Layer(s), 256 Units
   ⚠️ REMINDER: If you changed 'vocab_size' or 'seq_length', run PREPROCESS again!
3


Configuration explanation:

Seq_length : when using 20 tokens the model didn't have enough information to learn properly and understand the style of writing and order of words -> 100

max_vocab_size : when using 3000 or 7000 i got many <UNK> and extending the vocab helped with repetion and solving this problem.

drop_out rate: increasing it helped with overfitting

epocs: when i trained for more than 3 epocs the model started memorizing the data and giving perplexity less than 5(about 1-2). so I trained for less epocs

### 7. Execution: Train

Run the cell below to start the training pipeline. Monitor the `Train Loss` and `Val Loss`.



In [10]:
# @title 4. Run Preprocessing Pipeline
!python scripts/main.py preprocess

--- Starting Workflow for Experiment: 'sherlock_lstm_light_v1' ---
--- Step 1: Data Preprocessing ---
Starting Preprocessing...
Vocabulary size: 10000
total tokens: 656465
Preprocessing pipeline finished.
✅ Preprocessing completed successfully.


In [18]:
# @title Start Training
!python -u scripts/main.py train

--- Starting Workflow for Experiment: 'sherlock_lstm_light_v1' ---

--- Step 2: Model Training ---
[ProjectConfig] Saved configuration to /content/sherlock_project/checkpoints/config.json
Epoch 1 | Train Loss: 5.8384 | Val Loss: 4.2048
⭐ New best model saved at epoch 1
Epoch 2 | Train Loss: 3.3870 | Val Loss: 2.6641
⭐ New best model saved at epoch 2
Epoch 3 | Train Loss: 2.3328 | Val Loss: 1.9131
⭐ New best model saved at epoch 3
✅ Training completed successfully.


I realized that stopping in earlier epocs helped with perplexity. Also in other configurations in val_loss = 1.5-2 of best epoc i got the better perplexity and result.



### 8. Execution: Evaluation

**Metrics:**
* **Perplexity (PPL):** Lower is better. Measures how "surprised" the model is by the text.
* **Diversity (Distinct-N):** Higher is better. Measures how unique the generated vocabulary is.



In [19]:
# @title Calculate Perplexity
!python -u scripts/main.py evaluate

--- Starting Workflow for Experiment: 'sherlock_lstm_light_v1' ---

--- Step 3: Evaluating model from 'sherlock_lstm_light_v1'... ---
Evaluating Perplexity: 100% 42/42 [00:00<00:00, 82.85it/s]

Evaluating diversity on 5 samples...
  - Perplexity on validation set: 6.3415
  - Distinct-1 on generated sample: 0.9446
  - Distinct-2 on generated sample: 1.0000
✅ Evaluation completed successfully.


In [12]:
%%writefile scripts/evaluation.py
import sys
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
from nltk.util import ngrams
from typing import List, Tuple
import json

# Ensure the root directory is in the path for src imports
project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, project_root)

from src.config import ProjectConfig
from src.dataset import SherlockDataset
from src.model import EncoderDecoder
from src import utils

class ModelEvaluator:
    """
    Handles evaluation using Perplexity and Distinct-n metrics.
    Fixes the massive perplexity issue by ignoring padding in calculations.
    """
    def __init__(self, config: ProjectConfig):
        self.config = config
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

        # Consistent with your main.py and project structure
        self.experiment_dir = utils.get_experiment_dir(config)
        model_path = "/content/sherlock_project/checkpoints/best_model.pth"

        if not os.path.exists(model_path):
            raise FileNotFoundError(f"No model found at {model_path}. Train the model first.")

        # Load the vocabulary to get special token IDs
        with open(utils.VOCAB_PATH, 'r', encoding='utf-8') as f:
            self.vocab = torch.load(utils.VOCAB_PATH) if utils.VOCAB_PATH.endswith('.pth') else json.load(f)

        self.model = self._load_model(model_path)
        self.val_loader = self._get_val_loader()

    def _load_model(self, model_path: str) -> EncoderDecoder:
        model = EncoderDecoder(
            vocab_size=self.config.data.max_vocab_size,
            embedding_dim=self.config.model.embedding_dim,
            hidden_size=self.config.model.lstm_units,
            num_layers=self.config.model.num_layers,
            dropout_prob=self.config.model.dropout_rate,
        ).to(self.device)

        checkpoint = torch.load(model_path, map_location=self.device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        return model

    def _get_val_loader(self) -> DataLoader:
        """Fixes the 'DataConfig has no len()' error by loading the tensor first."""
        if not os.path.exists(utils.SEQUENCES_PATH):
            raise FileNotFoundError(f"Sequences not found at {utils.SEQUENCES_PATH}")

        # Load the actual data tensor (usually saved during preprocessing)
        data_tensor = torch.load(utils.SEQUENCES_PATH)

        # Initialize dataset with the tensor, NOT the config object
        full_dataset = SherlockDataset(data_tensor, self.config.data.sequence_length)

        # Use a fixed seed to ensure validation set is distinct from training
        train_size = int(0.8 * len(full_dataset))
        val_size = len(full_dataset) - train_size
        _, val_dataset = random_split(
            full_dataset, [train_size, val_size],
            generator=torch.Generator().manual_seed(42)
        )
        return DataLoader(val_dataset, batch_size=self.config.train.batch_size, shuffle=False)

    def calculate_perplexity(self) -> float:
        """Calculates token-level perplexity, ignoring the <PAD> index (0)."""
        # ignore_index=0 is critical for valid perplexity scores
        criterion = nn.CrossEntropyLoss(ignore_index=0, reduction='sum')
        total_loss = 0.0
        total_tokens = 0

        with torch.no_grad():
            for src, trg in tqdm(self.val_loader, desc="Evaluating Perplexity"):
                src, trg = src.to(self.device), trg.to(self.device)
                output, _ = self.model(src, trg)

                # Flatten the output and target for cross entropy
                loss = criterion(output.view(-1, output.size(-1)), trg.view(-1))
                total_loss += loss.item()
                # Only count non-padding tokens
                total_tokens += (trg != 0).sum().item()

        avg_loss = total_loss / total_tokens
        return torch.exp(torch.tensor(avg_loss)).item()

    @staticmethod
    def calculate_distinct_n(text: str) -> Tuple[float, float]:
        """Calculates unigram (D1) and bigram (D2) diversity."""
        tokens = text.lower().split()
        if not tokens: return 0.0, 0.0

        d1 = len(set(ngrams(tokens, 1))) / len(tokens)
        bigrams = list(ngrams(tokens, 2))
        d2 = len(set(bigrams)) / len(bigrams) if bigrams else 0.0
        return d1, d2

    def calculate_diversity_from_samples(self, generator_obj=None, prompts=None) -> Tuple[float, float]:
            """
            Calculates average Distinct-1 and Distinct-2 diversity.
            Modified to work without arguments to match the call in main.py.
            """
            # 1. Handle missing generator
            if generator_obj is None:
                from scripts.generate import TextGenerator
                # self.config is already available in ModelEvaluator's __init__
                generator_obj = TextGenerator(self.config)

            # 2. Handle missing prompts
            if prompts is None:
                prompts = [
                    "Sherlock Holmes was a",
                    "The mystery of the",
                    "London was a dark",
                    "He looked at the evidence",
                    "The case was finally"
                ]

            d1_scores = []
            d2_scores = []

            print(f"\nEvaluating diversity on {len(prompts)} samples...")

            for prompt in prompts:
                # Generate a sample using the 'sample' strategy for variety
                generated_text = generator_obj.generate(
                    prompt,
                    max_length=50,
                    strategy='sample'
                )

                # Calculate metrics
                d1, d2 = self.calculate_distinct_n(generated_text)
                d1_scores.append(d1)
                d2_scores.append(d2)

            avg_d1 = sum(d1_scores) / len(prompts)
            avg_d2 = sum(d2_scores) / len(prompts)

            return avg_d1, avg_d2

def evaluate(config: ProjectConfig):
    """Entry point called by main.py."""
    print("\n--- Step 3: Model Evaluation ---")
    evaluator = ModelEvaluator(config)

    # Perplexity
    ppl = evaluator.calculate_perplexity()
    print(f"✅ Perplexity on validation set: {ppl:.4f}")

    # Diversity
    from scripts.generate import TextGenerator
    generator = TextGenerator(config.experiment_name)
    sample_prompts = ["Sherlock Holmes was", "The case of the", "Watson looked at"]
    d1, d2 = evaluator.calculate_diversity_from_samples(generator, sample_prompts)

    print(f"✅ Distinct-1 (Diversity): {d1:.4f}")
    print(f"✅ Distinct-2 (Diversity): {d2:.4f}")







Overwriting scripts/evaluation.py


If the model were perfectly certain about every word said next, the perplexity would be 1. If it were guessing completely randomly from 10,000-word vocabulary, the perplexity would be 10,000.

calculate_perplexity:

This function iterates through the validation data. For every word, it calculates how "surprised" the model was by the actual word in the book.
Forward Pass: Asks the model to predict the next 50 words given a prompt.
Summing Loss: It sums up the total loss but only for actual words (ignoring index 0).

### 9. Execution: Story Generation

Test your Encoder-Decoder by providing prompts. The model will Encode your prompt and Decode a continuation.

In [13]:
%%writefile scripts/generate.py
import sys
import os
import re
import json
import torch
import torch.nn.functional as F
from typing import List, Tuple

# Fix Python path
project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, project_root)

from src.config import ProjectConfig
from src.model import EncoderDecoder
from src import utils
class TextGenerator:
    """Encapsulates the logic for generating text with the Encoder-Decoder model."""

    def __init__(self, config: ProjectConfig):
        self.config = config
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

        # Load paths consistent with ProjectConfig
        self.experiment_dir = utils.get_experiment_dir(config)
        model_path = "/content/sherlock_project/checkpoints/best_model.pth"

        # Load Vocabulary
        with open(utils.VOCAB_PATH, 'r', encoding='utf-8') as f:
            vocab_data = json.load(f)
            self.token_to_id = vocab_data["token_to_id"]
            self.id_to_token = {int(k): v for k, v in vocab_data["id_to_token"].items()}

        self.model = self._load_model(model_path)

    def _load_model(self, model_path: str) -> EncoderDecoder:
        model = EncoderDecoder(
            vocab_size=self.config.data.max_vocab_size,
            embedding_dim=self.config.model.embedding_dim,
            hidden_size=self.config.model.lstm_units,
            num_layers=self.config.model.num_layers,
            dropout_prob=self.config.model.dropout_rate,
        ).to(self.device)

        checkpoint = torch.load(model_path, map_location=self.device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        return model

    def _clean_prompt(self, text: str) -> List[int]:
        text = text.lower()
        text = re.sub(r"[^a-z0-9\s\.\,\!\?\;\:\'\-]", " ", text)
        tokens = text.split()
        return [self.token_to_id.get(t, self.token_to_id.get("<UNK>", 3)) for t in tokens]

    def generate(
        self,
        start_prompt: str,
        max_length: int = 50,
        strategy: str = 'sample',
        temperature: float = 0.7,
        repetition_penalty: float = 2, # Higher value = stronger repetition prevention
        top_k: int = 20                 # Limits options to most likely words
    ):
        """Generates text using the encoder-decoder hidden state."""
        with torch.no_grad():
            # 1. Encode context
            input_ids = self._clean_prompt(start_prompt)
            src_tensor = torch.tensor([input_ids], device=self.device)
            src_embedded = self.model.embedding(src_tensor)
            _, (h, c) = self.model.encoder_lstm(src_embedded)

            generated_words = []
            tokens_seen = []

            # Start with <SOS>
            curr_token = torch.tensor([[self.token_to_id.get("<SOS>", 1)]], device=self.device)
            state = (h, c)

            for _ in range(max_length):
                embedded = self.model.embedding(curr_token)
                output, state = self.model.decoder_lstm(embedded, state)
                logits = self.model.fc_out(output).squeeze(1) # (1, vocab_size)

                # --- Apply Repetition Penalty ---
                for token_id in set(tokens_seen):
                    if logits[0, token_id] > 0:
                        logits[0, token_id] /= repetition_penalty
                    else:
                        logits[0, token_id] *= repetition_penalty

                if strategy == 'greedy':
                    next_token = torch.argmax(logits, dim=-1).unsqueeze(0)
                else:
                    # --- Apply Top-K Filtering ---
                    indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
                    logits[indices_to_remove] = -float('Inf')

                    # Apply temperature
                    probs = F.softmax(logits / max(temperature, 1e-6), dim=-1)
                    next_token = torch.multinomial(probs, num_samples=1)

                token_id = next_token.item()

                # Stop if <EOS> is reached
                if token_id == self.token_to_id.get("<EOS>", 2):
                    break

                tokens_seen.append(token_id)
                word = self.id_to_token.get(token_id, "<UNK>")
                generated_words.append(word)
                curr_token = next_token

            return start_prompt + " " + " ".join(generated_words)

Overwriting scripts/generate.py


The Encoder reads the prompt and produces the "Context" (hidden/cell states).

The Decoder takes that context and predicts the first word.

That predicted word is then fed back into the Decoder as the input for the next word.

repetition penalty: This script keeps a list of tokens_seen. If the model tries to pick a word it has already used, the script manually lowers the "score" (logits) for that word, making it less likely to be chosen again.

Top_k filter : It looks at the top K (20) most likely words and sets the probability of every other word in the dictionary to negative infinity (−∞). This ensures the model only chooses from a pool of sensible candidates.

Low Temperature (< 1.0): Makes the model more "confident," usually resulting in safer, more common words.

In [20]:
# @title Generate Text
prompts = [
    "It was a dark and stormy night",
    "Sherlock Holmes sat in his chair",
    "The evidence clearly points to"
]

for p in prompts:
    print(f"\n📝 Prompt: {p}")
    !python -u scripts/main.py generate --prompt "{p}"


📝 Prompt: It was a dark and stormy night
--- Starting Workflow for Experiment: 'sherlock_lstm_light_v1' ---
Generating for: 'It was a dark and stormy night'
Result: It was a dark and stormy night take let mind small must passed since every when is face being to still hear side may quite mrs. young any any far possible door see man. little she whom where make back about by put entirely through already well, sir, know you then that they him, holmes, out am

📝 Prompt: Sherlock Holmes sat in his chair
--- Starting Workflow for Experiment: 'sherlock_lstm_light_v1' ---
Generating for: 'Sherlock Holmes sat in his chair'
Result: Sherlock Holmes sat in his chair too to he. but <UNK> no of across it, since fellow, such going for and white down every other took now, turned young any put by her never most not already through say these heard we has with man man, go see their stood was was would hand our all

📝 Prompt: The evidence clearly points to
--- Starting Workflow for Experiment: 'sherlock_l